# REQUIREMENTS

In [ ]:
!pip install cdt
!pip install gcastle

# LOAD DATASETS

In [ ]:
import networkx as nx
import numpy as np
from cdt.data import load_dataset
import pandas as pd
from castle.metrics import MetricsDAG
import random

In [ ]:

#############################
# 1) SACH - 11 Nodes
# #############################
s_data, s_graph = load_dataset('dream4-1')
loaded_data = s_data
#loaded_data = pd.read_csv('/content/SACHS.csv')

actual_dag = nx.to_numpy_array(s_graph, nodelist=s_data.columns)
loaded_data = loaded_data.to_numpy()
loaded_data.shape, actual_dag.shape

((100, 100), (100, 100))

In [ ]:
# SYNTHEITC DATASETS

loaded_data1 = pd.read_csv('/content/gaussian_50nodes.csv',header =0)
adj = pd.read_csv('/content/adj_matrix_50nodes_linear.csv',header =0)
loaded_data = loaded_data1.to_numpy()
actual_dag = adj.to_numpy()
loaded_data.shape, actual_dag.shape

((5000, 50), (50, 50))

# SYNC FRAMEWORK

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from scipy.linalg import expm as matrix_exponentialreg_type
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import pdist
import logging
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
import math
import networkx as nx
from torch.nn import TransformerEncoder, TransformerEncoderLayer




###############################
# 1) DAG Model (encoder)
###############################
class DAGModel(nn.Module):
    def __init__(self, data_dim, hidden_dim, nheads=8):
        super(DAGModel, self).__init__()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.data_dim = data_dim
        self.adj_dim = data_dim * data_dim
        self.hidden_dim = hidden_dim

        # Linear layer to project input data to `hidden_dim` for transformer
        self.data_projection = nn.Linear(data_dim, hidden_dim).to(self.device)

        # Transformer Encoder for processing data
        data_encoder_layer = TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nheads, dropout=0.2
        )
        self.data_encoder = TransformerEncoder(data_encoder_layer, num_layers=3).to(self.device)

        # Transformer Encoder for processing adjacency matrix
        adj_encoder_layer = TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nheads, dropout=0.2
        )
        self.adj_encoder = TransformerEncoder(adj_encoder_layer, num_layers=3).to(self.device)

        # MLPs to process adjacency features and combine
        self.adj_processor = nn.Sequential(
            nn.Linear(hidden_dim * data_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        ).to(self.device)

        self.combined_processor = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.adj_dim),
        ).to(self.device)

    def forward(self, data, prior_adj):
        """
        data: [batch_size, data_dim]  (the full batch, not the mean)
        prior_adj: [batch_size, data_dim, data_dim]
                  or [data_dim, data_dim] (we'll replicate if needed)
        """
        data = data.to(self.device)

        # 1) Project data to `hidden_dim`
        data_projected = self.data_projection(data)  # [B, hidden_dim]

        # Expand to sequence format for transformer
        batch_size = data.size(0)
        data_projected = data_projected.unsqueeze(1).expand(-1, self.data_dim, -1)  # [B, d, hidden_dim]
        x_data = self.data_encoder(data_projected)  # [B, d, hidden_dim]
        x_data = x_data.mean(dim=1)  # Aggregate along the sequence dimension [B, hidden_dim]

        # 2) Transformer Encoder on prior adjacency
        if prior_adj.dim() == 2:
            prior_adj = prior_adj.unsqueeze(0).expand(batch_size, -1, -1)
        prior_adj = prior_adj.to(self.device)

        # Project adjacency matrix to match `hidden_dim`
        prior_adj_projected = self.data_projection(prior_adj)  # Project to [B, d, hidden_dim]

        # Pass through adjacency transformer encoder
        x_adj = self.adj_encoder(prior_adj_projected)  # [B, d, hidden_dim]
        x_adj_flat = x_adj.view(batch_size, -1)  # [B, d*hidden_dim]
        x_adj_processed = self.adj_processor(x_adj_flat)  # [B, hidden_dim]

        # Combine data features + adjacency features
        x_combined = torch.cat([x_data, x_adj_processed], dim=1)  # [B, hidden_dim*2]

        # Predict adjacency logits
        adj_output = self.combined_processor(x_combined)   # [B, d*d]
        adj_output = adj_output.view(batch_size, self.data_dim, self.data_dim)
        return adj_output






#################################
# 2) Memory & ReinforceAgent
#################################
class ReinforceMemory:
    """
    A simpler memory class: we only store states, adj_matrices, log_probs, rewards, dones.
    We'll do a single update with them (no GAE, no advantage).
    """
    def __init__(self):
        self.states = []
        self.adj_matrices = []
        self.log_probs = []
        self.rewards = []
        self.dones = []

    def store_memory(self, state, adj_matrix, log_prob, reward, done):
        self.states.append(state)
        self.adj_matrices.append(adj_matrix)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.dones.append(done)

    def clear_memory(self):
        self.states = []
        self.adj_matrices = []
        self.log_probs = []
        self.rewards = []
        self.dones = []


class ReinforceAgent:

    def __init__(
        self,
        model,
        actor_lr,
        gamma,
        batch_size,
        partial_prior
    ):
        self.model = model
        self.gamma = gamma
        self.batch_size = batch_size
        self.partial_prior = partial_prior

        self.actor_optimizer = optim.AdamW(self.model.parameters(), lr=actor_lr)

        self.memory = ReinforceMemory()
        self.device = self.model.device

    def remember(self, state, adj_matrix, log_prob, reward, done):
        self.memory.store_memory(state, adj_matrix, log_prob, reward, done)

    def choose_action(self, state):
        """
        Now pass partial_prior to the model instead of random adjacency.
        """
        device = self.model.device
        if not isinstance(state, torch.Tensor):
            state = torch.tensor(state, dtype=torch.float32)
        state_tensor = state.unsqueeze(0).to(device)

        # forward pass with partial_prior
        adj_output = self.model(state_tensor, self.partial_prior)
        adj_probs = torch.sigmoid(adj_output)
        mask = 1.0 - torch.eye(self.model.data_dim, device=device)
        adj_probs = adj_probs * mask

        dist = torch.distributions.Bernoulli(probs=adj_probs)
        adj_sampled = dist.sample()
        log_prob = dist.log_prob(adj_sampled).sum()
        return adj_sampled.squeeze(0), log_prob


    def learn(self):
        """
        REINFORCE update:
          - For each transition, we have a log_prob and reward
          - actor_loss = -mean( log_prob[i] * reward[i] )
        We do not rely on advantage or multiple epochs here.
        """
        if len(self.memory.states) < self.batch_size:
            return None

        # Gather all
        log_probs = torch.stack(self.memory.log_probs)  # shape [N]
        rewards = torch.tensor(self.memory.rewards, dtype=torch.float32, device=self.device)

        # Actor loss
        actor_loss = -(log_probs * rewards).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
        self.actor_optimizer.step()

        self.memory.clear_memory()
        return actor_loss.item()



##################################################
# 3) The get_Reward class
##################################################

class get_Reward(object):
    _logger = logging.getLogger(__name__)

    def __init__(self, batch_num, maxlen, dim, inputdata, sl, su, lambda1_upper,
                 score_type='BIC', reg_type='LR', l1_graph_reg=0.0, verbose_flag=True):
        self.batch_num = batch_num
        self.maxlen = maxlen
        self.dim = dim
        self.baseint = 2**maxlen
        self.d = {}
        self.d_RSS = {}
        self.inputdata = inputdata
        self.n_samples = inputdata.shape[0]
        self.l1_graph_reg = l1_graph_reg
        self.verbose = verbose_flag
        self.sl = sl
        self.su = su
        self.lambda1_upper = lambda1_upper
        self.bic_penalty = np.log(inputdata.shape[0]) / inputdata.shape[0]

        if score_type not in ('BIC', 'BIC_different_var'):
            raise ValueError('Reward type not supported.')
        if reg_type not in ('LR', 'QR', 'GPR'):
            raise ValueError('Reg type not supported')
        self.score_type = score_type
        self.reg_type = reg_type

        self.ones = np.ones((inputdata.shape[0], 1), dtype=np.float32)
        self.poly = PolynomialFeatures()

    def cal_rewards(self, graphs, true_graph, lambda1, lambda2, lambda3):
        rewards_batches = []
        for graphi in graphs:
            reward_ = self.calculate_reward_single_graph(graphi, true_graph, lambda1, lambda2, lambda3)
            rewards_batches.append(reward_)
        return np.array(rewards_batches)

    def calculate_yerr(self, X_train, y_train):
        if self.reg_type == 'LR':
            return self.calculate_LR(X_train, y_train)
        elif self.reg_type == 'QR':
            return self.calculate_QR(X_train, y_train)
        elif self.reg_type == 'GPR':
            return self.calculate_GPR(X_train, y_train)
        else:
            assert False, 'Regressor not supported'

    def calculate_LR(self, X_train, y_train):
        X = np.hstack((X_train, self.ones))
        XtX = X.T.dot(X)
        Xty = X.T.dot(y_train)
        theta = np.linalg.solve(XtX, Xty)
        y_err = X.dot(theta) - y_train
        return y_err

    def calculate_QR(self, X_train, y_train):
        X_train = self.poly.fit_transform(X_train)[:, 1:]
        return self.calculate_LR(X_train, y_train)

    def calculate_GPR(self, X_train, y_train):
        med_w = np.median(pdist(X_train, 'euclidean'))
        gpr = GPR().fit(X_train / med_w, y_train)
        return y_train.reshape(-1, 1) - gpr.predict(X_train / med_w).reshape(-1, 1)

    def calculate_reward_single_graph(self, graph_batch, tgraph, lambda1, lambda2, lambda3):
        if isinstance(graph_batch, torch.Tensor):
            graph_batch = graph_batch.cpu().detach().numpy()
        if isinstance(tgraph, torch.Tensor):
            tgraph = tgraph.cpu().detach().numpy()

        # Simple mismatch penalty
        penalty = 0
        for i in range(self.maxlen):
            for j in range(self.maxlen):
                if tgraph[i][j] in [0, 1]:  # known edge or known non-edge
                    if graph_batch[i][j] != tgraph[i][j]:
                        penalty += 1

        # zero diagonal
        for i in range(self.maxlen):
            graph_batch[i][i] = 0

        graph_to_int = []
        graph_to_int2 = []
        for i in range(self.maxlen):
            tt = np.int32(graph_batch[i])
            graph_to_int2.append(int(''.join([str(ad) for ad in tt]), 2))
            graph_to_int.append(self.baseint * i + int(''.join([str(ad) for ad in tt]), 2))

        graph_batch_to_tuple = tuple(graph_to_int2)
        if graph_batch_to_tuple in self.d:
            score_cyc = self.d[graph_batch_to_tuple]
            return self.penalized_score(score_cyc, lambda1, lambda2, lambda3), score_cyc[0], score_cyc[1], score_cyc[2]

        # compute RSS
        RSS_ls = []
        for i in range(self.maxlen):
            col = graph_batch[i]
            if graph_to_int[i] in self.d_RSS:
                RSS_ls.append(self.d_RSS[graph_to_int[i]])
                continue

            if np.sum(col) < 0.1:
                y_err = self.inputdata[:, i]
                y_err = y_err - np.mean(y_err)
            else:
                cols_TrueFalse = col > 0.5
                X_train = self.inputdata[:, cols_TrueFalse]
                y_train = self.inputdata[:, i]
                y_err = self.calculate_yerr(X_train, y_train)

            RSSi = np.sum(np.square(y_err))
            if self.reg_type == 'GPR':
                RSSi += 1.0
            RSS_ls.append(RSSi)
            self.d_RSS[graph_to_int[i]] = RSSi

        if self.score_type == 'BIC':
            BIC = np.log(np.sum(RSS_ls)/self.n_samples+1e-8) \
                  + np.sum(graph_batch)*self.bic_penalty/self.maxlen
        elif self.score_type == 'BIC_different_var':
            BIC = np.sum(np.log(np.array(RSS_ls)/self.n_samples+1e-8)) \
                  + np.sum(graph_batch)*self.bic_penalty

        score = self.score_transform(BIC)
        cycness = np.trace(matrix_exponentialreg_type(np.array(graph_batch))) - self.maxlen
        reward = -(score + lambda1*float(cycness > 1e-5) + lambda2*cycness + lambda3*penalty)

        if self.l1_graph_reg > 0:
            reward += self.l1_graph_reg * np.sum(graph_batch)
            score  += self.l1_graph_reg * np.sum(graph_batch)

        self.d[graph_batch_to_tuple] = (score, cycness, penalty)
        return reward, score, cycness, penalty

    def score_transform(self, s):
        return (s - self.sl) / (self.su - self.sl) * self.lambda1_upper

    def penalized_score(self, score_cyc, lambda1, lambda2, lambda3):
        score, cyc, penalty = score_cyc
        return -(score + lambda1*float(cyc>1e-5) + lambda2*cyc + lambda3*penalty)

    def update_scores(self, score_cycs, lambda1, lambda2, lambda3):
        ls = []
        for score_cyc in score_cycs:
            ls.append(self.penalized_score(score_cyc, lambda1, lambda2, lambda3))
        return ls

    def update_all_scores(self, lambda1, lambda2, lambda3):
        score_cycs = list(self.d.items())
        ls = []
        for graph_int, score_cyc in score_cycs:
            ls.append((
                graph_int,
                (
                    self.penalized_score(score_cyc, lambda1, lambda2, lambda3),
                    score_cyc[0],
                    score_cyc[1]
                )
            ))
        return sorted(ls, key=lambda x: x[1][0])


############################
# 4) The Trainer (example)
############################
def train_dag_model_with_reinforce(model, data,partial_prior,partial_prior1 ,num_epochs, actor_lr):
    """
    A simpler REINFORCE-like approach using ReinforceAgent above.
    We do NOT do advantage, GAE, or separate critic updates.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # Hyperparams
    batch_size = 64
    gamma = 0.99

    # Agent
    agent = ReinforceAgent(
        model=model,
        actor_lr=actor_lr,
        gamma=gamma,
        batch_size=batch_size,
        partial_prior=torch.tensor(partial_prior, dtype=torch.float32, device=device)
    )

    # Just an example reward calculator
    reward_calculator = get_Reward(
        batch_num=1,
        maxlen=model.data_dim,
        dim=1,
        inputdata=data,
        sl=0,
        su=1,
        lambda1_upper=1,
        score_type='BIC_different_var',
        reg_type='LR',
        l1_graph_reg=1.0,
        verbose_flag=False
    )

    best_reward = -float("inf")
    best_adj_matrix = None

    all_rewards = []
    actor_loss_history = []
    edges_over_time = []
    cycles_over_time = []

    from collections import deque
    env_states = deque(data)

    for epoch in tqdm(range(num_epochs)):
        max_steps = 100
        step_count = 0
        done = False
        state = env_states[0]
        episode_reward = 0.0

        while not done:
            # choose action
            adj_matrix, log_prob = agent.choose_action(state)

            # threshold adjacency
            adj_probs = torch.sigmoid(adj_matrix)
            mask_eye = 1.0 - torch.eye(model.data_dim, device=adj_probs.device)
            adj_probs = adj_probs * mask_eye
            adj_matrix = (adj_probs >= 0.7).float()  # or any threshold
            # If you want cycle removal:
            adj_matrix = ensure_acyclic(adj_matrix)

            # reward
            reward, score, cyc,penalty= reward_calculator.calculate_reward_single_graph(
                adj_matrix,partial_prior1, lambda1=1.0, lambda2=2.0, lambda3=0.5
            )
            #print(score)
            #print(sum)
            #print(cyc)
            #reward = -(2*score+ 0.5*sum)
            #print('Reward before:', reward)
            #reward = max(min(reward, -1),1)
            #print('Reward after:', reward)
            #if abs(reward) > 1e6:  # Arbitrary threshold for abnormally large values
                #print("Abnormal Reward Detected:", reward)
                #print(adj_matrix)



            if reward > best_reward:
                best_reward = reward
                best_adj_matrix = adj_matrix.detach().cpu().numpy().copy()
                best_adj_probs = adj_probs.detach().cpu().numpy().copy()

            episode_reward += reward

            # memory
            agent.remember(
                torch.tensor(state, dtype=torch.float32).to(device),
                adj_matrix.to(device),
                log_prob.to(device),
                reward,
                done
            )

            step_count += 1
            if step_count >= max_steps or step_count >= len(data):
                done = True
            else:
                next_idx = step_count % len(data)
                state = env_states[next_idx]

        all_rewards.append(episode_reward)

        # Once we have enough transitions in memory => update
        loss_val = agent.learn()
        if loss_val is not None:
            actor_loss_history.append(loss_val)

        # Edges/cycles logging
        if best_adj_matrix is not None:
            last_adj_t = torch.tensor(best_adj_matrix, dtype=torch.float32, device=device)
            num_edges = last_adj_t.sum().item()
            cyc_val = np.trace(matrix_exponentialreg_type(last_adj_t.cpu().numpy())) - model.data_dim
            edges_over_time.append(num_edges)
            cycles_over_time.append(cyc_val)

        if epoch % 2 == 0:
            print(f"Epoch {epoch}, EpisodeReward={episode_reward:.4f}, BestReward={best_reward:.4f}")

    print("=== Finished Training ===")
    print(f"Best adjacency reward found: {best_reward:.4f}")

    # Simple final plot
    plt.figure(figsize=(16, 4))
    plt.subplot(1, 3, 1)
    plt.plot(all_rewards)
    plt.title("Episode Reward")

    plt.subplot(1, 3, 2)
    plt.plot(actor_loss_history)
    plt.title("Actor Loss")

    plt.subplot(1, 3, 3)
    plt.plot(edges_over_time, label="Edges (Best so far)")
    plt.plot(cycles_over_time, label="Cycles (Best so far)")
    plt.legend()
    plt.title("Edges & Cycles")
    plt.tight_layout()
    plt.savefig("reinforce_training_progress.png")
    plt.close()

    return model, best_adj_matrix, best_adj_probs

# HELPER FUNCTIONS

In [ ]:

##################################################
# 1) RemoveCycles: ensure_acyclic
##################################################

def ensure_acyclic(adj_matrix: torch.Tensor) -> torch.Tensor:
    """
    Removes a randomly selected lowest-weight edge in each cycle until no cycles remain.
    """
    if adj_matrix.dim() != 2:
        raise ValueError("ensure_acyclic expects a 2D adjacency matrix.")

    device = adj_matrix.device
    A = adj_matrix.detach().cpu().numpy()
    N = A.shape[0]

    G = nx.DiGraph()
    for i in range(N):
        for j in range(N):
            if i != j and A[i, j] > 0:
                G.add_edge(i, j, weight=A[i, j])

    while True:
        try:
            cycle_edges = nx.find_cycle(G, orientation='original')
            # Find all edges with the minimum weight
            min_edges = []
            min_w = float('inf')
            for (u, v, direction) in cycle_edges:
                w_ = G[u][v]['weight']
                if w_ < min_w:
                    min_edges = [(u, v)]
                    min_w = w_
                elif w_ == min_w:
                    min_edges.append((u, v))
            # Randomly pick one edge from the minimum-weight edges
            min_edge = random.choice(min_edges)
            G.remove_edge(min_edge[0], min_edge[1])
            A[min_edge[0], min_edge[1]] = 0.0
        except nx.NetworkXNoCycle:
            break

    return torch.from_numpy(A).float().to(device)



##################################################
# 2) PruneWeakEdges: Pruning
##################################################
def calculate_threshold(W, d):
    flattened = W.flatten()
    sorted_weights = np.sort(flattened)[::-1]
    d_idx = min(d-1, len(sorted_weights)-1)
    return sorted_weights[d_idx]


def graph_prunned_by_coef(graph_batch, X):
    d = len(graph_batch)
    reg = LinearRegression()
    W = []

    for i in range(d):
        col = np.abs(graph_batch[i]) > 0.5
        if np.sum(col) == 0:
            W.append(np.zeros(d))
            continue

        X_train = X[:, col]
        y = X[:, i]
        reg.fit(X_train, y)
        reg_coeff = reg.coef_

        new_reg_coeff = np.zeros(d, dtype=float)
        parent_indices = np.where(col)[0]
        for idx_parent, coef_val in zip(parent_indices, reg_coeff):
            new_reg_coeff[idx_parent] = coef_val

        W.append(new_reg_coeff)

    W = np.array(W)
    th = calculate_threshold(W, X.shape[1])
    pruned = (np.abs(W) >= th).astype(np.float32)
    return pruned

##################################################
# 3) Partial Prior
##################################################

def create_partial_prior(actual_dag: np.ndarray, fraction=0.25) -> np.ndarray:
    """
    Takes a ground-truth adjacency 'actual_dag' (d x d) with 1 => edge, 0 => no edge.
    Returns 'prior_adj':
      - prior_adj[i,j] = 1 => we are certain there's an edge i->j
      - prior_adj[i,j] = 0 => we are certain there's no edge i->j (optional, not used here)
      - prior_adj[i,j] = -1 => unknown
    We'll keep 'fraction' of the edges as known=1, the rest are unknown=-1.
    """
    d = actual_dag.shape[0]
    prior_adj = -1 * np.ones((d, d), dtype=int)

    edges = np.argwhere(actual_dag > 0.5)
    np.random.shuffle(edges)
    num_edges = len(edges)
    keep_count = int(fraction * num_edges)

    known_edges = edges[:keep_count]
    for (i, j) in known_edges:
        prior_adj[i, j] = 1  # we are sure there's an edge i->j

    # if you also want to fix known no-edges, do similarly here
    return prior_adj

##################################################
# 3) Metrics
##################################################

def count_accuracy(B_true, B_est):
    d = B_true.shape[0]

    # Predicted edges and undirected edges
    pred_und = np.flatnonzero(B_est == -1)
    pred = np.flatnonzero(B_est == 1)
    cond = np.flatnonzero(B_true)
    cond_reversed = np.flatnonzero(B_true.T)
    cond_skeleton = np.concatenate([cond, cond_reversed])

    # True positives (TP)
    true_pos = np.intersect1d(pred, cond, assume_unique=True)
    true_pos_und = np.intersect1d(pred_und, cond_skeleton, assume_unique=True)
    true_pos = np.concatenate([true_pos, true_pos_und])

    # False positives (FP)
    false_pos = np.setdiff1d(pred, cond_skeleton, assume_unique=True)
    false_pos_und = np.setdiff1d(pred_und, cond_skeleton, assume_unique=True)
    false_pos = np.concatenate([false_pos, false_pos_und])

    # Reverse edges
    extra = np.setdiff1d(pred, cond, assume_unique=True)
    reverse = np.intersect1d(extra, cond_reversed, assume_unique=True)

    # Predicted size and condition negatives
    pred_size = len(pred) + len(pred_und)
    cond_neg_size = 0.5 * d * (d - 1) - len(cond)

    # False discovery rate (FDR), true positive rate (TPR), false positive rate (FPR)
    fdr = float(len(reverse) + len(false_pos)) / max(pred_size, 1)
    tpr = float(len(true_pos)) / max(len(cond), 1)
    fpr = float(len(reverse) + len(false_pos)) / max(cond_neg_size, 1)

    # True negatives (TN) and false negatives (FN)
    pred_lower = np.flatnonzero(np.tril(B_est + B_est.T))
    cond_lower = np.flatnonzero(np.tril(B_true + B_true.T))
    extra_lower = np.setdiff1d(pred_lower, cond_lower, assume_unique=True)
    missing_lower = np.setdiff1d(cond_lower, pred_lower, assume_unique=True)

    # SHD: structural hamming distance
    shd = len(extra_lower) + len(missing_lower) + len(reverse)

    # FN and TN calculations
    fn = len(cond) - len(true_pos)
    tn = cond_neg_size - len(false_pos)

    # True negative rate (TNR) and false negative rate (FNR)
    tnr = float(tn) / max(cond_neg_size, 1)
    fnr = float(fn) / max(len(cond), 1)

    return {
        'fdr': fdr,
        'tpr': tpr,
        'fpr': fpr,
        'tnr': tnr,
        'fnr': fnr,
        'tp': len(true_pos),
        'tn': tn,
        'fp': len(false_pos),
        'fn': fn,
        'shd': shd,
        'nnz': pred_size
    }


# TRAINER

In [ ]:
def predict_dag_with_reinforce_no_threshold(
    model,
    data,
    partial_prior, # <--- ADD Generative_prior here
    partial_prior1, # <--- ADD partial_prior here
    num_epochs=10,
    actor_lr=1e-3
):
    """
    1) Train the model with REINFORCE and get the best adjacency (no thresholding).
    2) Prune edges with `graph_prunned_by_coef`.
    3) Ensure acyclicity if needed.
    4) Return final adjacency as NumPy array.
    """
    # 1) Train the model via your REINFORCE function, now passing partial_prior
    trained_model, best_adj_matrix, best_adj_probs = train_dag_model_with_reinforce(
        model=model,
        data=data,
        partial_prior=actual_dag,
        partial_prior1 = partial_prior1,  # pass it along
        num_epochs=num_epochs,
        actor_lr=actor_lr
    )

    pruned_np = graph_prunned_by_coef(best_adj_matrix, data)

    # 3) Ensure final adjacency is acyclic
    pruned_pt = ensure_acyclic(torch.tensor(pruned_np, dtype=torch.float32))
    final_dag_np = pruned_pt.cpu().numpy()

    print("Final DAG shape:", final_dag_np.shape)
    print("Sum of edges in the final DAG:", final_dag_np.sum())
    return final_dag_np, trained_model,best_adj_matrix,best_adj_probs


# MAIN

In [ ]:



if __name__ == "__main__":
    import numpy as np
    import torch

    d = loaded_data.shape[1]
    data = loaded_data
    partial_prior = np.load('add path to generative prior') # Generative Prior
    partial_prior1 = create_partial_prior(actual_dag, fraction=0.25) # Prior Knowledge Graph


    # Initialize your DAG model
    model = DAGModel(data_dim=d, hidden_dim=64)

    # Train with REINFORCE, get the final DAG without thresholding
    final_dag_np, trained_model, ba,best_adj_probs = predict_dag_with_reinforce_no_threshold(
        model=model,
        data=data,
        num_epochs=10,
        actor_lr=1e-3,
        partial_prior = partial_prior,
        partial_prior1= partial_prior1
        )



    # Evaluate
    final_dag = graph_prunned_by_coef(best_adj_probs, loaded_data)
    final_dag = ensure_acyclic(torch.tensor(final_dag, dtype=torch.float32))
    accuracy_results = count_accuracy(actual_dag, final_dag)
    print("Accuracy results:", accuracy_results)
